# Notebook 3: FiLM World Model — Evaluation

Evaluate the trained FiLM model on the 2025 test set.
- Delta-to-price reconstruction
- MSPE / RMSPE metrics (matching XGB baseline methodology)
- Per-step and per-commodity breakdown
- Ablation: with vs without events

**Environment**: Kaggle GPU. Requires `stage1_best.pt` and pre-computed embeddings.

In [ ]:
!pip install -q transformers accelerate peft

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from bisect import bisect_left, bisect_right
from dataclasses import dataclass

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

## Config & Data (same as Notebook 2)

In [ ]:
@dataclass
class Config:
    data_dir: str = "/kaggle/input/finance-world-model-dataset"
    embeddings_path: str = "/kaggle/input/event-embeddings/event_embeddings.pt"
    model_path: str = "/kaggle/input/film-model-checkpoints/stage1_best.pt"
    n_features: int = 46
    d_model: int = 256
    lookback: int = 10
    horizon: int = 10
    n_transformer_layers: int = 2
    n_heads: int = 4
    ff_dim: int = 1024
    dropout: float = 0.1
    max_events_per_window: int = 8
    qwen_hidden: int = 2560

    critical_features: tuple = (
        "YF_Price_Crude Oil (WTI)",
        "YF_Price_Crude Oil (Brent)",
        "YF_Price_Gold",
        "YF_Price_Natural Gas",
        "YF_Price_Silver",
        "YF_Price_Copper",
    )
    diff_features: tuple = ("FRED_DFF", "FRED_DGS10", "FRED_T10YIE")
    test_start: str = "2025-01-01"
    test_end: str = "2025-12-31"


cfg = Config()

In [ ]:
# Load market data & compute log returns (identical to Notebook 2)
market_df = pd.read_csv(
    f"{cfg.data_dir}/combined_commodity_data.csv",
    parse_dates=["date"], index_col="date",
)
feature_cols = market_df.columns.tolist()
critical_indices = [feature_cols.index(f) for f in cfg.critical_features]

diff_cols = [c for c in feature_cols if c in cfg.diff_features]
log_cols = [c for c in feature_cols if c not in cfg.diff_features]

deltas_df = pd.DataFrame(index=market_df.index[1:], columns=feature_cols, dtype=np.float32)
for col in log_cols:
    vals = market_df[col].values.astype(np.float64)
    vals[vals == 0] = np.nan
    vals = pd.Series(vals).ffill().bfill().values
    deltas_df[col] = np.diff(np.log(vals)).astype(np.float32)
for col in diff_cols:
    deltas_df[col] = np.diff(market_df[col].values).astype(np.float32)
deltas_df = deltas_df.clip(-0.5, 0.5).fillna(0.0)

# Load events
events_df = pd.read_csv(f"{cfg.data_dir}/gdelt_events_with_market_impact.csv")
events_df["event_time"] = pd.to_datetime(events_df["event_time"])
events_df = events_df[events_df["market_impact"] == 1].reset_index(drop=True)

event_dates_sorted = events_df["event_time"].sort_values().values
event_idx_by_date = events_df.sort_values("event_time").index.values
event_timestamps = [pd.Timestamp(d) for d in event_dates_sorted]

def find_events_in_window(start_date, end_date):
    left = bisect_left(event_timestamps, pd.Timestamp(start_date))
    right = bisect_right(event_timestamps, pd.Timestamp(end_date))
    return event_idx_by_date[left:right].tolist()

print(f"Market: {market_df.shape}, Deltas: {deltas_df.shape}, Events: {len(events_df)}")

In [ ]:
# Load embedding cache
cache_data = torch.load(cfg.embeddings_path, weights_only=False)
embedding_cache = cache_data["embeddings"]

## Model & Dataset (copied from Notebook 2)

In [ ]:
# --- Model definitions (identical to Notebook 2 Section B) ---

class MarketMLP(nn.Module):
    def __init__(self, n_features, d_model):
        super().__init__()
        self.linear = nn.Linear(n_features, d_model)
        self.act = nn.GELU()
    def forward(self, x):
        return self.act(self.linear(x))

class FiLMLayer(nn.Module):
    def __init__(self, conditioning_dim, feature_dim):
        super().__init__()
        self.gamma_linear = nn.Linear(conditioning_dim, feature_dim)
        self.beta_linear = nn.Linear(conditioning_dim, feature_dim)
        nn.init.ones_(self.gamma_linear.bias)
        nn.init.zeros_(self.gamma_linear.weight)
        nn.init.zeros_(self.beta_linear.bias)
        nn.init.zeros_(self.beta_linear.weight)
    def forward(self, x, conditioning):
        gamma = self.gamma_linear(conditioning).unsqueeze(1)
        beta = self.beta_linear(conditioning).unsqueeze(1)
        return gamma * x + beta

class EventEncoder(nn.Module):
    def __init__(self, qwen_hidden, d_model):
        super().__init__()
        self.projection = nn.Linear(qwen_hidden, d_model)
        self.no_event_embed = nn.Parameter(torch.zeros(d_model))
    def forward(self, event_embeds, event_mask):
        B = event_embeds.shape[0]
        e_reprs = self.projection(event_embeds)
        valid_mask = ~event_mask
        n_valid = valid_mask.sum(dim=1, keepdim=True).clamp(min=1)
        masked_reprs = e_reprs * valid_mask.unsqueeze(-1).float()
        e_global = masked_reprs.sum(dim=1) / n_valid.float()
        no_events = valid_mask.sum(dim=1) == 0
        if no_events.any():
            e_global[no_events] = self.no_event_embed.unsqueeze(0)
        return e_global, e_reprs

class EventMarketCrossAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        self.cross_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
    def forward(self, market_seq, event_reprs, event_mask):
        attn_out, _ = self.cross_attn(query=market_seq, key=event_reprs, value=event_reprs, key_padding_mask=event_mask)
        return self.norm(market_seq + attn_out)

class TemporalTransformer(nn.Module):
    def __init__(self, d_model, n_heads, n_layers, ff_dim, dropout, seq_len):
        super().__init__()
        self.pos_encoding = nn.Parameter(torch.randn(1, seq_len, d_model) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=ff_dim,
            dropout=dropout, activation="gelu", batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
    def forward(self, x):
        return self.encoder(x + self.pos_encoding)

class PredictionHead(nn.Module):
    def __init__(self, d_model, n_features, critical_indices):
        super().__init__()
        self.head = nn.Linear(d_model, n_features)
        self.register_buffer("critical_indices", torch.tensor(critical_indices, dtype=torch.long))
    def forward(self, h, target_deltas=None):
        predicted = self.head(h)
        loss = None
        if target_deltas is not None:
            loss = F.mse_loss(predicted[:, :, self.critical_indices], target_deltas[:, :, self.critical_indices])
        return predicted, loss

class FiLMWorldModel(nn.Module):
    def __init__(self, cfg, critical_indices):
        super().__init__()
        d = cfg.d_model
        self.market_mlp = MarketMLP(cfg.n_features, d)
        self.film_anchor = FiLMLayer(cfg.n_features, d)
        self.event_encoder = EventEncoder(cfg.qwen_hidden, d)
        self.film_event = FiLMLayer(d, d)
        self.cross_attn = EventMarketCrossAttention(d, cfg.n_heads, cfg.dropout)
        self.temporal_transformer = TemporalTransformer(d, cfg.n_heads, cfg.n_transformer_layers, cfg.ff_dim, cfg.dropout, cfg.lookback)
        self.prediction_head = PredictionHead(d, cfg.n_features, critical_indices)
    def forward(self, market_deltas, anchor_state, event_embeds, event_mask, target_deltas=None):
        h = self.market_mlp(market_deltas)
        h = self.film_anchor(h, anchor_state)
        e_global, e_reprs = self.event_encoder(event_embeds, event_mask)
        h = self.film_event(h, e_global)
        h = self.cross_attn(h, e_reprs, event_mask)
        h = self.temporal_transformer(h)
        return self.prediction_head(h, target_deltas)

In [ ]:
# --- Dataset (identical to Notebook 2 Section A) ---

class CommodityEventDataset(Dataset):
    def __init__(self, deltas_df, raw_df, events_df, cfg, start_date, end_date, embedding_cache=None):
        self.cfg = cfg
        self.embedding_cache = embedding_cache
        mask = (deltas_df.index >= start_date) & (deltas_df.index <= end_date)
        self.deltas = deltas_df[mask].values.astype(np.float32)
        self.dates = deltas_df[mask].index
        raw_mask = (raw_df.index >= start_date) & (raw_df.index <= end_date)
        self.raw_values = raw_df[raw_mask].values.astype(np.float32)
        self.raw_dates = raw_df[raw_mask].index
        self.n_samples = max(0, len(self.deltas) - cfg.lookback - cfg.horizon + 1)
        print(f"Dataset [{start_date} to {end_date}]: {self.n_samples} samples")

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        L, H = self.cfg.lookback, self.cfg.horizon
        market_deltas = torch.from_numpy(self.deltas[idx:idx+L])
        target_deltas = torch.from_numpy(self.deltas[idx+L:idx+L+H])
        anchor_date = self.dates[idx+L-1]
        raw_idx = self.raw_dates.get_loc(anchor_date)
        anchor_state = torch.from_numpy(self.raw_values[raw_idx])

        event_indices = find_events_in_window(self.dates[idx], self.dates[idx+L-1])
        event_indices = event_indices[:self.cfg.max_events_per_window]
        n_events = len(event_indices)
        max_k = self.cfg.max_events_per_window

        event_embeds = torch.zeros(max_k, self.cfg.qwen_hidden)
        if self.embedding_cache is not None:
            for i, eidx in enumerate(event_indices):
                event_embeds[i] = self.embedding_cache[eidx]

        event_mask = torch.ones(max_k, dtype=torch.bool)
        event_mask[:n_events] = False

        return {
            "market_deltas": market_deltas,
            "anchor_state": anchor_state,
            "target_deltas": target_deltas,
            "event_embeds": event_embeds,
            "event_mask": event_mask,
            "n_events": n_events,
            "anchor_date": str(anchor_date.date()),
        }

def collate_fn(batch):
    return {
        "market_deltas": torch.stack([s["market_deltas"] for s in batch]),
        "anchor_state": torch.stack([s["anchor_state"] for s in batch]),
        "target_deltas": torch.stack([s["target_deltas"] for s in batch]),
        "event_embeds": torch.stack([s["event_embeds"] for s in batch]),
        "event_mask": torch.stack([s["event_mask"] for s in batch]),
        "anchor_dates": [s["anchor_date"] for s in batch],
    }

## Load Model & Test Data

In [ ]:
# Load model
model = FiLMWorldModel(cfg, critical_indices).to(DEVICE)
model.load_state_dict(torch.load(cfg.model_path, map_location=DEVICE, weights_only=True))
model.eval()
print("Model loaded.")

# Test dataset
test_ds = CommodityEventDataset(
    deltas_df, market_df, events_df, cfg,
    cfg.test_start, cfg.test_end, embedding_cache,
)
test_loader = DataLoader(
    test_ds, batch_size=64, shuffle=False,
    collate_fn=collate_fn, num_workers=2,
)

## Run Inference & Reconstruct Prices

In [ ]:
# ============================================================
# Delta-to-price reconstruction
# ============================================================

def deltas_to_prices(predicted_deltas, anchor_prices, feature_indices, feature_cols, diff_features):
    """
    Reconstruct absolute prices from predicted log-return deltas.

    For log-return features: price_{t+k} = anchor * exp(cumsum(deltas[0:k]))
    For diff features: price_{t+k} = anchor + cumsum(deltas[0:k])

    Args:
        predicted_deltas: (H, n_features) numpy array
        anchor_prices: (n_features,) numpy array
        feature_indices: list of int, which features to reconstruct
        feature_cols: list of str, all column names
        diff_features: set of str, features using simple difference
    Returns:
        prices: (H, len(feature_indices)) numpy array
    """
    H = predicted_deltas.shape[0]
    prices = np.zeros((H, len(feature_indices)))

    for j, fi in enumerate(feature_indices):
        col = feature_cols[fi]
        anchor = anchor_prices[fi]
        cum_deltas = np.cumsum(predicted_deltas[:, fi])

        if col in diff_features:
            prices[:, j] = anchor + cum_deltas
        else:
            prices[:, j] = anchor * np.exp(cum_deltas)

    return prices

In [ ]:
# ============================================================
# Collect predictions on test set
# ============================================================

all_pred_deltas = []   # list of (H, n_features) arrays
all_target_deltas = []
all_anchor_prices = []
all_anchor_dates = []

with torch.no_grad():
    for batch in test_loader:
        market_deltas = batch["market_deltas"].to(DEVICE)
        anchor_state = batch["anchor_state"].to(DEVICE)
        event_embeds = batch["event_embeds"].to(DEVICE)
        event_mask = batch["event_mask"].to(DEVICE)
        target_deltas = batch["target_deltas"]

        pred, _ = model(market_deltas, anchor_state, event_embeds, event_mask)
        pred = pred.cpu().numpy()

        for i in range(pred.shape[0]):
            all_pred_deltas.append(pred[i])
            all_target_deltas.append(target_deltas[i].numpy())
            all_anchor_prices.append(batch["anchor_state"][i].numpy())
            all_anchor_dates.append(batch["anchor_dates"][i])

print(f"Test samples: {len(all_pred_deltas)}")

In [ ]:
# ============================================================
# Compute MSPE/RMSPE per window and per step
# ============================================================

diff_features_set = set(cfg.diff_features)
n_windows = len(all_pred_deltas)
n_critical = len(critical_indices)
H = cfg.horizon

# Per-window, per-step, per-feature SPE
spe_all = np.zeros((n_windows, H, n_critical))  # squared percentage error

for w in range(n_windows):
    pred_prices = deltas_to_prices(
        all_pred_deltas[w], all_anchor_prices[w],
        critical_indices, feature_cols, diff_features_set,
    )
    actual_prices = deltas_to_prices(
        all_target_deltas[w], all_anchor_prices[w],
        critical_indices, feature_cols, diff_features_set,
    )

    # Avoid division by zero
    actual_safe = np.where(np.abs(actual_prices) < 1e-8, 1e-8, actual_prices)
    spe_all[w] = ((pred_prices - actual_prices) / actual_safe) ** 2

# Aggregate metrics
mspe_per_window = spe_all.mean(axis=(1, 2))  # (n_windows,)
rmspe_per_step = np.sqrt(spe_all.mean(axis=(0, 2))) * 100  # (H,)
rmspe_per_feature = np.sqrt(spe_all.mean(axis=(0, 1))) * 100  # (n_critical,)
overall_rmspe = np.sqrt(spe_all.mean()) * 100

print(f"Overall RMSPE: {overall_rmspe:.3f}%")
print(f"Average MSPE: {mspe_per_window.mean():.6f}")
print(f"\nPer-step RMSPE:")
for step in range(H):
    print(f"  Day {step+1}: {rmspe_per_step[step]:.3f}%")
print(f"\nPer-commodity RMSPE:")
for i, f in enumerate(cfg.critical_features):
    print(f"  {f}: {rmspe_per_feature[i]:.3f}%")

## Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. Per-step RMSPE
ax = axes[0, 0]
ax.bar(range(1, H + 1), rmspe_per_step, color="steelblue")
ax.set_xlabel("Forecast Horizon (days)")
ax.set_ylabel("RMSPE (%)")
ax.set_title("Error by Forecast Horizon")
ax.set_xticks(range(1, H + 1))

# 2. Per-window MSPE distribution
ax = axes[0, 1]
ax.hist(mspe_per_window, bins=30, color="coral", edgecolor="white")
ax.axvline(mspe_per_window.mean(), color="red", linestyle="--", label=f"Mean: {mspe_per_window.mean():.4f}")
ax.set_xlabel("MSPE")
ax.set_ylabel("Count")
ax.set_title("Per-Window MSPE Distribution")
ax.legend()

# 3. Per-commodity RMSPE
ax = axes[1, 0]
short_names = [f.replace("YF_Price_", "") for f in cfg.critical_features]
ax.barh(short_names, rmspe_per_feature, color="mediumseagreen")
ax.set_xlabel("RMSPE (%)")
ax.set_title("Error by Commodity")

# 4. Cumulative error over time
ax = axes[1, 1]
rolling_mspe = pd.Series(mspe_per_window).rolling(30, min_periods=1).mean()
ax.plot(rolling_mspe.values, color="purple", linewidth=1)
ax.set_xlabel("Test Window Index")
ax.set_ylabel("30-window Rolling MSPE")
ax.set_title("Error Over Time (Test Set)")

plt.suptitle(f"FiLM World Model — Test RMSPE: {overall_rmspe:.3f}%", fontsize=14)
plt.tight_layout()
plt.show()

## Sample Trajectory Plots

In [ ]:
# Plot a few example trajectories for WTI and Gold
sample_windows = [0, len(all_pred_deltas) // 4, len(all_pred_deltas) // 2, 3 * len(all_pred_deltas) // 4]
plot_features = [(0, "Crude Oil (WTI)"), (2, "Gold")]  # index into critical features

fig, axes = plt.subplots(len(plot_features), len(sample_windows), figsize=(20, 8))

for row, (feat_idx, feat_name) in enumerate(plot_features):
    col_idx = critical_indices[feat_idx]
    for col, w in enumerate(sample_windows):
        ax = axes[row, col]

        pred_p = deltas_to_prices(
            all_pred_deltas[w], all_anchor_prices[w],
            critical_indices, feature_cols, diff_features_set,
        )[:, feat_idx]
        actual_p = deltas_to_prices(
            all_target_deltas[w], all_anchor_prices[w],
            critical_indices, feature_cols, diff_features_set,
        )[:, feat_idx]

        days = range(1, H + 1)
        anchor = all_anchor_prices[w][col_idx]
        ax.axhline(anchor, color="gray", linestyle=":", alpha=0.5)
        ax.plot(days, actual_p, "o-", label="Actual", markersize=3)
        ax.plot(days, pred_p, "s--", label="Predicted", markersize=3, alpha=0.8)
        ax.set_title(f"{feat_name} — {all_anchor_dates[w]}", fontsize=9)
        ax.set_xlabel("Day")
        if col == 0:
            ax.set_ylabel(f"{feat_name} Price")
        ax.legend(fontsize=7)

plt.suptitle("Sample 10-Day Forecast Trajectories", fontsize=13)
plt.tight_layout()
plt.show()

## Ablation: With vs Without Events

In [ ]:
# ============================================================
# Run inference with events zeroed out (ablation)
# ============================================================

all_pred_deltas_no_events = []

with torch.no_grad():
    for batch in test_loader:
        market_deltas = batch["market_deltas"].to(DEVICE)
        anchor_state = batch["anchor_state"].to(DEVICE)
        # Zero out events and mask all
        event_embeds = torch.zeros_like(batch["event_embeds"]).to(DEVICE)
        event_mask = torch.ones_like(batch["event_mask"]).to(DEVICE)  # all masked

        pred, _ = model(market_deltas, anchor_state, event_embeds, event_mask)
        pred = pred.cpu().numpy()

        for i in range(pred.shape[0]):
            all_pred_deltas_no_events.append(pred[i])

# Compute RMSPE without events
spe_no_events = np.zeros((n_windows, H, n_critical))
for w in range(n_windows):
    pred_prices = deltas_to_prices(
        all_pred_deltas_no_events[w], all_anchor_prices[w],
        critical_indices, feature_cols, diff_features_set,
    )
    actual_prices = deltas_to_prices(
        all_target_deltas[w], all_anchor_prices[w],
        critical_indices, feature_cols, diff_features_set,
    )
    actual_safe = np.where(np.abs(actual_prices) < 1e-8, 1e-8, actual_prices)
    spe_no_events[w] = ((pred_prices - actual_prices) / actual_safe) ** 2

rmspe_no_events = np.sqrt(spe_no_events.mean()) * 100
rmspe_per_feature_no_events = np.sqrt(spe_no_events.mean(axis=(0, 1))) * 100

print(f"\n{'='*50}")
print(f"ABLATION: Event Contribution")
print(f"{'='*50}")
print(f"With events:    RMSPE = {overall_rmspe:.3f}%")
print(f"Without events: RMSPE = {rmspe_no_events:.3f}%")
print(f"Event lift:     {rmspe_no_events - overall_rmspe:.3f}% improvement")
print(f"\nPer-commodity:")
for i, f in enumerate(cfg.critical_features):
    diff = rmspe_per_feature_no_events[i] - rmspe_per_feature[i]
    print(f"  {f}: {rmspe_per_feature[i]:.3f}% -> {rmspe_per_feature_no_events[i]:.3f}% (lift: {diff:+.3f}%)")

In [ ]:
# Ablation comparison chart
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(n_critical)
width = 0.35

ax.bar(x - width/2, rmspe_per_feature, width, label="With Events", color="steelblue")
ax.bar(x + width/2, rmspe_per_feature_no_events, width, label="Without Events", color="coral")
ax.set_xticks(x)
ax.set_xticklabels(short_names, rotation=15)
ax.set_ylabel("RMSPE (%)")
ax.set_title("Ablation: Event Contribution by Commodity")
ax.legend()
plt.tight_layout()
plt.show()